In [32]:
!pip install openai numpy voyageai


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
!pip install python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import numpy as np
from openai import OpenAI

In [35]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env

api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY not found in .env file")

voyage_api_key = os.getenv("VOYAGE_API_KEY")

if voyage_api_key is None:
    raise ValueError("VOYAGE_API_KEY not found in .env file")

In [36]:
client = OpenAI(api_key=api_key)

In [37]:
import voyageai

voy_client = voyageai.Client(api_key=voyage_api_key)

In [38]:
# Technical (medical jargon)
technical_sentence = "The patient exhibited severe diaphoresis and tachycardia"

# Layman equivalent
layman_query = "The person was sweating heavily with a fast heart rate"

In [39]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding)

def get_voyage_embedding(text):
    response = voy_client.embed(
        texts=[text],
        model="voyage-2"   # you can also try: voyage-large-2
    )
    return np.array(response.embeddings[0])

def get_voyage_embeddings_batch(texts):
    response = voy_client.embed(
        texts=texts,   # 🔥 pass full list
        model="voyage-2"
    )
    return [np.array(e) for e in response.embeddings]

def get_openai_embeddings_batch(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [np.array(e.embedding) for e in response.data]

tech_embedding = get_embedding(technical_sentence)
query_embedding = get_embedding(layman_query)

print("Technical Embedding Shape:", tech_embedding.shape)
print("Query Embedding Shape:", query_embedding.shape)

voy_tech_embedding = get_voyage_embedding(technical_sentence)
voy_query_embedding = get_voyage_embedding(layman_query)

print("Voyage Technical Embedding Shape:", voy_tech_embedding.shape)
print("Voyage Query Embedding Shape:", voy_query_embedding.shape)

Technical Embedding Shape: (1536,)
Query Embedding Shape: (1536,)
Voyage Technical Embedding Shape: (1024,)
Voyage Query Embedding Shape: (1024,)


In [40]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity_score = cosine_similarity(tech_embedding, query_embedding)
voy_similarity = cosine_similarity(voy_tech_embedding, voy_query_embedding)

print("Voyage Cosine Similarity Score:", voy_similarity)
print("Cosine Similarity Score:", similarity_score)

Voyage Cosine Similarity Score: 0.8279624084903082
Cosine Similarity Score: 0.6391173111109092


In [41]:
documents = [
    # Core target (should ideally match)
    "The patient exhibited severe diaphoresis and tachycardia",

    # Medical jargon (related)
    "The individual presented with tachyarrhythmia and profuse sweating",
    "Clinical signs included hyperhidrosis and elevated heart rate",
    "Patient showed signs of cardiovascular stress and sweating",
    "Observed symptoms include palpitations and excessive perspiration",
    "The subject had sinus tachycardia and diaphoresis",
    "Autonomic instability with sweating and rapid pulse",
    "Elevated pulse rate accompanied by sweating episodes",
    "Excessive sweating with increased sympathetic activity",
    "Cardiac acceleration and sweating were observed",

    # Layman equivalents (semantic matches)
    "The person was sweating heavily and had a fast heartbeat",
    "He felt very sweaty and his heart was racing",
    "She was sweating a lot with a rapid pulse",
    "Heavy sweating and increased heart rate were noticed",
    "The individual had a fast heart rate and was drenched in sweat",
    "Rapid heartbeat along with excessive sweating",
    "The person felt hot, sweaty, and their heart was pounding",
    "Sweating a lot and heart beating very fast",
    "Profuse sweating with heart racing quickly",

    # Partial overlaps
    "The patient had chest pain and rapid breathing",
    "Shortness of breath and elevated pulse",
    "Increased heart rate without sweating",
    "Sweating but normal heart rate",
    "The patient felt dizzy with a rapid pulse",
    "Mild perspiration with elevated blood pressure",
    "The subject experienced palpitations",
    "Heart rate increased during exercise",
    "Sweating due to high temperature",

    # Other medical conditions (distractors)
    "The patient had a mild headache",
    "The patient experienced nausea and vomiting",
    "The patient had normal vital signs",
    "The patient reported abdominal pain",
    "The patient had a fever and chills",
    "Signs of dehydration were observed",
    "Patient diagnosed with hypertension",
    "The subject had diabetes mellitus",
    "Chronic kidney disease was noted",
    "The patient suffered from asthma",
    "The patient had anemia",
    "The patient showed signs of infection",
    "The patient had joint pain",
    "The patient complained of fatigue",
    "The patient had blurred vision",
    "The patient had a sore throat",
    "The patient had muscle weakness",
    "The patient experienced seizures",
    "The patient had skin rashes",
    "The patient had insomnia",

    # Completely irrelevant (noise)
    "The car engine overheated during the trip",
    "The weather was extremely humid today",
    "He ran a marathon in record time",
    "The stock market showed rapid growth",
    "She cooked a delicious meal",
    "The dog was barking loudly",
    "The meeting was postponed",
    "The laptop battery drained quickly",
    "He was sweating after intense exercise",
    "The room temperature was very high",
    "She felt nervous before the exam",
    "The athlete had a fast pulse after running",
    "The machine was operating at high speed",
    "The server experienced high load",
    "The system showed increased latency",
    "The sun was very hot that day",
    "He felt anxious and restless",
    "The crowd was excited and energetic",
    "The air was thick and humid",
    "The workout caused heavy sweating"
]

doc_embeddings = get_openai_embeddings_batch(documents)
voy_doc_embeddings = get_voyage_embeddings_batch(documents)

In [42]:
query = "The person was sweating heavily with a fast heart rate"
query_vec = get_embedding(query)

scores = []

for i, doc_vec in enumerate(doc_embeddings):
    score = cosine_similarity(query_vec, doc_vec)
    scores.append((documents[i], score))

# Sort by similarity
scores.sort(key=lambda x: x[1], reverse=True)

for doc, score in scores:
    print(f"{score:.4f} -> {doc}")

0.9620 -> The person was sweating heavily and had a fast heartbeat
0.8758 -> The individual had a fast heart rate and was drenched in sweat
0.8366 -> She was sweating a lot with a rapid pulse
0.7863 -> The person felt hot, sweaty, and their heart was pounding
0.7858 -> Profuse sweating with heart racing quickly
0.7774 -> Sweating a lot and heart beating very fast
0.7618 -> He felt very sweaty and his heart was racing
0.7430 -> Heavy sweating and increased heart rate were noticed
0.7283 -> He was sweating after intense exercise
0.7227 -> Rapid heartbeat along with excessive sweating
0.7055 -> The individual presented with tachyarrhythmia and profuse sweating
0.6867 -> Patient showed signs of cardiovascular stress and sweating
0.6839 -> Elevated pulse rate accompanied by sweating episodes
0.6610 -> The workout caused heavy sweating
0.6544 -> Cardiac acceleration and sweating were observed
0.6461 -> Autonomic instability with sweating and rapid pulse
0.6430 -> Sweating due to high tempera

In [44]:
voy_query_vec = get_voyage_embedding(query)

voy_scores = []

for i, doc_vec in enumerate(voy_doc_embeddings):
    score = cosine_similarity(voy_query_vec, doc_vec)
    voy_scores.append((documents[i], score))

# Sort
voy_scores.sort(key=lambda x: x[1], reverse=True)

for doc, score in voy_scores:
    print(f"{score:.4f} -> {doc}")

0.9914 -> The person was sweating heavily and had a fast heartbeat
0.9610 -> The individual had a fast heart rate and was drenched in sweat
0.9333 -> The person felt hot, sweaty, and their heart was pounding
0.9321 -> She was sweating a lot with a rapid pulse
0.9275 -> Heavy sweating and increased heart rate were noticed
0.9268 -> He felt very sweaty and his heart was racing
0.9143 -> Cardiac acceleration and sweating were observed
0.9136 -> The individual presented with tachyarrhythmia and profuse sweating
0.9110 -> Rapid heartbeat along with excessive sweating
0.9039 -> Profuse sweating with heart racing quickly
0.9030 -> Patient showed signs of cardiovascular stress and sweating
0.8965 -> Increased heart rate without sweating
0.8938 -> Sweating a lot and heart beating very fast
0.8890 -> He was sweating after intense exercise
0.8844 -> Sweating but normal heart rate
0.8831 -> Elevated pulse rate accompanied by sweating episodes
0.8732 -> The athlete had a fast pulse after running
0.

In [45]:
print("""
Observation:

- Check if the top result matches the technical sentence.
- If similarity is not very high, it shows:
    → General embedding model struggles with domain-specific vocabulary.

Key Insight:

- "diaphoresis" ≠ "sweating" strongly in general embeddings
- "tachycardia" ≠ "fast heart rate" strongly

This demonstrates:
→ Domain gap problem
→ Need for domain-specific embeddings
""")


Observation:

- Check if the top result matches the technical sentence.
- If similarity is not very high, it shows:
    → General embedding model struggles with domain-specific vocabulary.

Key Insight:

- "diaphoresis" ≠ "sweating" strongly in general embeddings
- "tachycardia" ≠ "fast heart rate" strongly

This demonstrates:
→ Domain gap problem
→ Need for domain-specific embeddings



In [46]:
for i, (doc, score) in enumerate(scores):
    print(f"Rank {i+1}")
    print("Document:", doc)
    print("Similarity:", score)
    print("-" * 50)

Rank 1
Document: The person was sweating heavily and had a fast heartbeat
Similarity: 0.9619921777810412
--------------------------------------------------
Rank 2
Document: The individual had a fast heart rate and was drenched in sweat
Similarity: 0.8758492203122639
--------------------------------------------------
Rank 3
Document: She was sweating a lot with a rapid pulse
Similarity: 0.8366490224052572
--------------------------------------------------
Rank 4
Document: The person felt hot, sweaty, and their heart was pounding
Similarity: 0.7863347454371212
--------------------------------------------------
Rank 5
Document: Profuse sweating with heart racing quickly
Similarity: 0.7857501604041977
--------------------------------------------------
Rank 6
Document: Sweating a lot and heart beating very fast
Similarity: 0.7774056212605952
--------------------------------------------------
Rank 7
Document: He felt very sweaty and his heart was racing
Similarity: 0.7617903415852759
-------

In [47]:
print("Top 10 — OpenAI Model")
for i in range(10):
    print(f"{i+1}. {scores[i][0]} ({scores[i][1]:.4f})")

print("\n" + "="*60 + "\n")

print("Top 10 — Voyage Model")
for i in range(10):
    print(f"{i+1}. {voy_scores[i][0]} ({voy_scores[i][1]:.4f})")

Top 10 — OpenAI Model
1. The person was sweating heavily and had a fast heartbeat (0.9620)
2. The individual had a fast heart rate and was drenched in sweat (0.8758)
3. She was sweating a lot with a rapid pulse (0.8366)
4. The person felt hot, sweaty, and their heart was pounding (0.7863)
5. Profuse sweating with heart racing quickly (0.7858)
6. Sweating a lot and heart beating very fast (0.7774)
7. He felt very sweaty and his heart was racing (0.7618)
8. Heavy sweating and increased heart rate were noticed (0.7430)
9. He was sweating after intense exercise (0.7283)
10. Rapid heartbeat along with excessive sweating (0.7227)


Top 10 — Voyage Model
1. The person was sweating heavily and had a fast heartbeat (0.9914)
2. The individual had a fast heart rate and was drenched in sweat (0.9610)
3. The person felt hot, sweaty, and their heart was pounding (0.9333)
4. She was sweating a lot with a rapid pulse (0.9321)
5. Heavy sweating and increased heart rate were noticed (0.9275)
6. He felt 

In [48]:
target = "The patient exhibited severe diaphoresis and tachycardia"

# OpenAI rank
openai_rank = [i for i, (doc, _) in enumerate(scores) if doc == target][0] + 1

# Voyage rank
voyage_rank = [i for i, (doc, _) in enumerate(voy_scores) if doc == target][0] + 1

print("Clinical Sentence Rank:")
print("OpenAI Model Rank:", openai_rank)
print("Voyage Model Rank:", voyage_rank)

Clinical Sentence Rank:
OpenAI Model Rank: 20
Voyage Model Rank: 30


In [49]:
print("""
FINAL ANALYSIS — Phase 1

1. General Model (OpenAI):
- Strong at surface-level semantic similarity
- Favors layman-to-layman matches
- Struggles mapping:
    diaphoresis → sweating
    tachycardia → fast heart rate

2. Domain-Adapted Model (Voyage):
- Better alignment between clinical and layman language
- Improved ranking of medical terminology
- Captures deeper semantic equivalence

CONCLUSION:
→ No single embedding model works for all use cases
→ Domain-specific or domain-adapted models outperform general models in specialized contexts
""")


FINAL ANALYSIS — Phase 1

1. General Model (OpenAI):
- Strong at surface-level semantic similarity
- Favors layman-to-layman matches
- Struggles mapping:
    diaphoresis → sweating
    tachycardia → fast heart rate

2. Domain-Adapted Model (Voyage):
- Better alignment between clinical and layman language
- Improved ranking of medical terminology
- Captures deeper semantic equivalence

CONCLUSION:
→ No single embedding model works for all use cases
→ Domain-specific or domain-adapted models outperform general models in specialized contexts

